In [0]:
from pyspark.sql.functions import col, to_timestamp, when, expr, current_timestamp

In [0]:
tabela = spark.read.table("`ct-esteira-dados-aviacao`.raw.fat_voos_raw_dep_arr")

In [0]:
# Padronizando tipo e alterando dados de colunas 

fat_voos_trusted_dep_arr = (
    tabela
    .withColumn('partida_prevista', to_timestamp(col("partida_prevista"), "dd/MM/yyyy HH:mm"))
    .withColumn('partida_real', to_timestamp(col("partida_real"), "dd/MM/yyyy HH:mm"))
    .withColumn('chegada_prevista', to_timestamp(col("chegada_prevista"), "dd/MM/yyyy HH:mm"))
    .withColumn('chegada_real', to_timestamp(col('chegada_real'), "dd/MM/yyyy HH:mm"))
    .withColumn('numero_de_assentos', col('numero_de_assentos').cast("int"))
    .withColumn('load_data', expr("current_timestamp() - INTERVAL 3 HOURS"))
    .withColumn(
        "empresa_aerea",
        when(col("sigla_icao_empresa_aerea").contains("GLO"), "GOL")
        .when(col("sigla_icao_empresa_aerea").contains("AZU"), "AZUL BRASIL")
        .when(col("sigla_icao_empresa_aerea").contains("ACN"), "AZUL CONECTA")
        .when(col("sigla_icao_empresa_aerea").contains("LAN"), "LATAM BRASIL")
        .otherwise(col("empresa_aerea"))
    )
    )

In [0]:
tabela_voos.printSchema()

In [0]:
fat_voos_trusted_dep_arr.write \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .partitionBy("sigla_icao_empresa_aerea") \
    .saveAsTable("`ct-esteira-dados-aviacao`.trusted.fat_voos_trusted_dep_arr")